# Data completeness of games_matched_final.xlsx

Which attributes of the file of Navid have data, which are empty, and where the same information is written in
more than one column. Functions: `completeness.py`. The map of the attributes and the proposed rules: `fields.yaml`.

**Counted as missing**: empty, or written as N/A, none, -, null, unknown. A `0` is counted apart.
**Status**: matched (confirmed), fuzzy (not confirmed), unmatched (no candidate). The data of matched and fuzzy can be used; fuzzy is marked.

**Output**: `output/completeness_report.xlsx`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd() if (Path.cwd() / "completeness.py").exists() else Path.cwd() / "catalog_project" / "check_games_matched_final"
sys.path[:0] = [str(HERE), str(HERE.parent)]
import completeness as cp  # noqa: E402
import master_lib as ml  # noqa: E402

FILE = HERE.parent / "games_matched_final.xlsx"
OUT = HERE / "output"
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220, "display.max_rows", 80)

## 1. Load

The three sheets of games as one table (2,951 games) and the sheet `crawler_records` (one row per game and website).
Takes about 15 seconds.

In [ ]:
games, records = cp.load(FILE)
fields = cp.load_fields()
assert cp.uncovered_columns(games, records, fields) == {"games": [], "crawler_records": []}, "a column is not in fields.yaml"
FILE_DATE = pd.Timestamp(FILE.stat().st_mtime, unit="s").date()
print(f"{FILE.name} of {FILE_DATE}: {len(games)} games, {games.shape[1]} columns | crawler_records {len(records)} rows")
games["overall_status"].value_counts()

## 2. Coverage by field

One row per attribute. `attr_*` is the column of Navid. `union` is every place with a value: `attr_*`, the other
columns of the file and `crawler_records`. Level on matched + fuzzy: sufficient from 80 %, partial from 50 %, below that needs enrichment.

In [ ]:
field_table = cp.field_coverage(games, records, fields)
field_table[["field", "kind", "attr_matched_%", "union_matched_%", "union_usable_%", "only_outside_attr", "conflicts", "level"]]

## 3. Scattered data

Games that have a value in one place while the column of Navid (`attr_*`) is empty, by status.
If they are fuzzy or unmatched, `attr_*` may leave them out on purpose. If they are matched, it is a gap that can be filled.

In [ ]:
scattered = cp.dispersion(games, records, fields)
examples = cp.dispersion_examples(games, records, fields)
scattered.sort_values("attr_empty_matched", ascending=False).head(12)

## 4. How the values are written

For each field, the values or the shapes (digits are 9, letters are a) by column. It shows what has to be normalized
before the columns can be unified.

In [ ]:
formats = cp.value_formats(games, records, fields)
columns = cp.column_coverage(games, records, fields)
print(len(formats), "rows of formats |", int((columns["written_as_NA"] > 0).sum()), "columns with values written as N/A |", int((columns["zeros"] > 0).sum()), "columns with zeros")
formats[formats["field"] == "volatility"].groupby("column").head(5)

## 5. Proposed rules to unify

One row per field: order of the sources, normalization and conflict rule. It is a proposal to validate with Navid.

In [ ]:
rules = cp.unification_rules(fields, field_table)
rules[["field", "level", "order", "normalize"]].head(10)

## 6. Report

In [ ]:
usable = games["overall_status"].isin(cp.USABLE)
read_me = pd.DataFrame(list({
    "file": f"{FILE.name}, of {FILE_DATE}",
    "built on": str(pd.Timestamp.today().date()),
    "games": f"{len(games)}: " + ", ".join(f"{s} {n}" for s, n in games["overall_status"].value_counts().items()),
    "crawler_records": f"{len(records)} rows, one per game and website",
    "counted as missing": "empty, or written as N/A, NA, N-A, none, -, null, unknown, [] or {}. A 0 is counted apart (columns: zeros)",
    "usable data": "matched (confirmed) and fuzzy (not confirmed). Unmatched games have no confirmed candidate",
    "union": "every place that holds the field: the column attr_ of Navid, the other columns of the file and crawler_records",
    "level": "sufficient from 80% of matched + fuzzy; partial from 50%; needs enrichment below; empty at 0",
    "fields": "one row per attribute: coverage of attr_ and of the union, games recoverable from other columns, conflicts, level",
    "columns": "every column of the file (the games and crawler_records) with its coverage by status",
    "dispersion": "games with a value outside the column attr_ of Navid, by place and status",
    "dispersion_examples": "the first 20 matched games of each field with the values found, to check by hand",
    "formats": "how the values of each field are written (values or shapes), to write the normalization",
    "unification_rules": "proposed rule of each field, to validate with Navid",
    "third party text": "description, review_text, pros, cons, faqs, game_text, sections and blog_articles are text of third party websites: the team has not decided if they enter the catalog",
}.items()), columns=["item", "value"])
ml.write_excel(OUT / "completeness_report.xlsx", {
    "read me": read_me, "fields": field_table, "columns": columns, "dispersion": scattered, "dispersion_examples": examples,
    "formats": formats, "unification_rules": rules},
    freeze={"fields": "B2", "columns": "C2", "dispersion": "B2", "dispersion_examples": "B2", "formats": "B2", "unification_rules": "B2"},
    wide={"fields": {"by_source_%": 70}, "dispersion_examples": {"values_found": 90, "game_name": 28}, "formats": {"example": 40},
          "unification_rules": {"order": 30, "normalize": 90, "conflict": 40, "rows_used": 40}, "read me": {"value": 120}})
print(f"{len(field_table)} fields | {len(columns)} columns -> {OUT / 'completeness_report.xlsx'}")
field_table["level"].value_counts()